In [1]:
import numpy as np 



In [2]:
data_1e6 = np.load('/export/data/vgiusepp/odisseo_data/data_fix_position/sbi-sim/data/sbi-benchmarks/odisseo/x_1000000.npy')

In [3]:
!pip install concurrent.futures

ERROR: Could not find a version that satisfies the requirement concurrent.futures (from versions: none)
ERROR: No matching distribution found for concurrent.futures


In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed


In [5]:
def process_sample(sample):
    """Compute 3 histograms from 1 sample"""
    bins = [64, 32]

    ph1_phi2, _, _ = np.histogram2d(sample[1], sample[2], bins=bins, range=[[-120., 70.], [-8, 2]])
    R_vR, _, _ = np.histogram2d(sample[0], sample[3], bins=bins, range=[[6., 20.], [-250., 250.]])
    vphicosphi2_vphi2, _, _ = np.histogram2d(sample[4], sample[5], bins=bins, range=[[-2., 1.], [-0.1, 0.1]])

    return np.stack([ph1_phi2, R_vR, vphicosphi2_vphi2], axis=0)  # Shape: (3, 64, 32)


def calculate_histogram_stats_parallel(data, max_workers=8):
    """Parallel histogram stats computation: mean/std over log1p histograms"""
    all_histograms = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_sample, sample) for sample in data]
        for future in as_completed(futures):
            all_histograms.append(future.result())

    all_histograms = np.stack(all_histograms)  # Shape: (N_samples, 3, 64, 32)
    all_histograms = np.log1p(all_histograms)

    # Calculate mean and std for each channel independently
    mean_per_channel = np.mean(all_histograms, axis=(0, 2, 3))  # Shape: (3,)
    std_per_channel = np.std(all_histograms, axis=(0, 2, 3))    # Shape: (3,)

    return mean_per_channel, std_per_channel

# Usage in your notebook:
mean_histogram_1e6, std_histogram_1e6 = calculate_histogram_stats_parallel(data_1e6)
np.savez('/export/data/vgiusepp/odisseo_data/data_fix_position/preprocess/mean_std_log_1e6.npz', 
         mean_x=mean_histogram_1e6, 
         std_x=std_histogram_1e6)

In [6]:
print('mean histogram', mean_histogram_1e6)
print('std histogram', std_histogram_1e6)

mean histogram [0.00098003 0.00035404 0.00034501]
std histogram [0.02686971 0.0156627  0.01547433]


In [11]:
# Calculate statistics for each histogram type separately
def calculate_histogram_stats(data):
    """Calculate mean and std for each histogram channel independently"""
    bins = [64, 32]
    all_histograms = []
    
    for sample in data:
        ph1_phi2, _, _ = np.histogram2d(sample[1], sample[2], bins=bins, range=[[-120., 70.], [-8, 2]])
        R_vR, _, _ = np.histogram2d(sample[0], sample[3], bins=bins, range=[[6., 20.], [-250., 250.]])
        vphicosphi2_vphi2, _, _ = np.histogram2d(sample[4], sample[5], bins=bins, range=[[-2., 1.], [-0.1, 0.1]])
        histograms = np.stack([ph1_phi2, R_vR, vphicosphi2_vphi2], axis=0)
        all_histograms.append(histograms)
    
    # all_histograms = np.log1p(all_histograms)  # Shape: (N_samples, 3, 64, 32)
    
    # Calculate mean and std for each channel independently
    mean_per_channel = np.mean(all_histograms, axis=(0, 2, 3))  # Shape: (3,)
    std_per_channel = np.std(all_histograms, axis=(0, 2, 3))    # Shape: (3,)

    return mean_per_channel, std_per_channel

# Usage in your notebook:
# mean_histogram_1e6_2, std_histogram_1e6_2 = calculate_histogram_stats(data_1e6)
np.savez('/export/data/vgiusepp/odisseo_data/data_fix_position/preprocess/mean_std_channel_1e6.npz', 
         mean_x=mean_histogram_1e6_2, 
         std_x=std_histogram_1e6_2)

In [10]:
print('mean histogram', mean_histogram_1e6_2)
print('std histogram', std_histogram_1e6_2)

mean histogram [0.00145455 0.00051081 0.00049815]
std histogram [0.04060171 0.02259976 0.02235672]


In [5]:
print('hello')

hello


In [4]:

print(mean_histogram_1e6.shape, std_histogram_1e6.shape)

(3, 64, 32) (3, 64, 32)


In [5]:
data_1e5 = np.load('/export/data/vgiusepp/odisseo_data/data_fix_position/sbi-sim/data/sbi-benchmarks/odisseo/x_100000.npy')

mean_histogram_1e5, std_histogram_1e5 = calculate_histogram_stats(data_1e5)

print(mean_histogram_1e5.shape, std_histogram_1e5.shape)

(3, 64, 32) (3, 64, 32)


In [18]:
(mean_histogram_1e6 - mean_histogram_1e5).mean()

np.float64(6.445312499999983e-07)

In [17]:
(std_histogram_1e6 - std_histogram_1e5).mean()

np.float64(0.00042155051943389236)

In [1]:
import matplotlib.pyplot as plt
plt.hist(std_histogram_1e6.flatten(), bins=100, range=(0, 0.5))

NameError: name 'std_histogram_1e6' is not defined

In [4]:
import jax.numpy as jnp

mean_x = jnp.load('/export/data/vgiusepp/odisseo_data/data_fix_position/preprocess/mean_std_1e6.npz')['std_x']
print(mean_x)

[[[0.00387295 0.00374163 0.00299999 ... 0.00299999 0.00374163 0.00282842]
  [0.00360553 0.00244948 0.00282842 ... 0.00244948 0.00346408 0.00331661]
  [0.00435886 0.00299999 0.00360553 ... 0.00374163 0.00346408 0.00282842]
  ...
  [0.00264574 0.00223606 0.002      ... 0.00282842 0.00264574 0.00244948]
  [0.00282842 0.00264574 0.00223606 ... 0.002      0.00316226 0.00223606]
  [0.00223606 0.001      0.00173205 ... 0.00173205 0.00223606 0.002     ]]

 [[0.         0.         0.00223606 ... 0.00141421 0.00141421 0.001     ]
  [0.         0.         0.00141421 ... 0.001      0.001      0.        ]
  [0.         0.         0.00244948 ... 0.00141421 0.         0.        ]
  ...
  [0.         0.001      0.002      ... 0.001      0.         0.        ]
  [0.         0.001      0.00173205 ... 0.00141421 0.         0.        ]
  [0.         0.001      0.002      ... 0.         0.         0.        ]]

 [[0.00316226 0.00360553 0.00299999 ... 0.00173205 0.00331661 0.00244948]
  [0.00264574 0.003464

# Normalization of Point Cloud 

In [1]:
import numpy as np

In [ ]:
data_1e5 = np.load('/export/data/vgiusepp/odisseo_data/data_fix_position/sbi-sim/data/sbi-benchmarks/odisseo/x_100000.npy')
data_1e5 = data_1e5.reshape(-1, 6)
print(data_1e5.shape)

mean_x = np.mean(data_1e5, axis=0)
std_x = np.std(data_1e5, axis=0)

print("Mean and standard deviation for 1e5 point cloud data:")
print("Mean:", mean_x)
print("Standard Deviation:", std_x)
np.savez('/export/data/vgiusepp/odisseo_data/data_fix_position/preprocess/mean_std_1e5_pointcloud.npz',
         mean_x=mean_x, std_x=std_x)

In [6]:
mean_x = np.mean(data_1e5, axis=0)
std_x = np.std(data_1e5, axis=0)

print("Mean and standard deviation for 1e5 point cloud data:")
print("Mean:", mean_x)
print("Standard Deviation:", std_x)
np.savez('/export/data/vgiusepp/odisseo_data/data_fix_position/preprocess/mean_std_1e5_pointcloud.npz',
         mean_x=mean_x, std_x=std_x)

Mean and standard deviation for 1e5 point cloud data:
Mean: [ 9.54696758e+00 -2.62425200e+01 -2.20864214e+00 -3.73875109e+01
 -8.36267021e-01 -2.45720135e-02]
Standard Deviation: [  5.13866914  43.76636714   7.02531742 100.2366489    1.08460315
   0.49360626]


In [2]:
data_1e6 = np.load('/export/data/vgiusepp/odisseo_data/data_fix_position/sbi-sim/data/sbi-benchmarks/odisseo/x_1000000.npy')
data_1e6 = data_1e6.reshape(-1, 6)
print(data_1e6.shape)

mean_x = np.mean(data_1e6, axis=0)
std_x = np.std(data_1e6, axis=0)

print("Mean and standard deviation for 1e6 point cloud data:")
print("Mean:", mean_x)
print("Standard Deviation:", std_x)
np.savez('/export/data/vgiusepp/odisseo_data/data_fix_position/preprocess/mean_std_1e6_pointcloud.npz',
         mean_x=mean_x, std_x=std_x)


(1000000000, 6)
Mean and standard deviation for 1e6 point cloud data:
Mean: [ 9.54769695e+00 -2.62339777e+01 -2.22726333e+00 -3.72828818e+01
 -8.36731688e-01 -2.47061010e-02]
Standard Deviation: [  5.13112746  43.85714956   7.08194019 100.33506808   1.3669442
   0.83776806]


# POint cloud varying position

In [1]:
import numpy as np

In [5]:
data_1e4 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position/sbi-sim/data/sbi-benchmarks/odisseo_AllParametersPosition/x_10000.npy')
data_1e4 = data_1e4.reshape(-1, 6)
print(data_1e4.shape)

mean_x = np.mean(data_1e4, axis=0)
std_x = np.std(data_1e4, axis=0)

print("Mean and standard deviation for 1e4 point cloud data:")
print("Mean:", mean_x)
print("Standard Deviation:", std_x)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position/preprocess/mean_std_1e4_pointcloud.npz',
         mean_x=mean_x, std_x=std_x)

(10000000, 6)
Mean and standard deviation for 1e4 point cloud data:
Mean: [ 1.00938185e+01 -2.39800480e+01 -1.54237320e+00 -5.69424583e+01
 -7.32295885e-01 -2.76011829e-02]
Standard Deviation: [ 4.84295294 41.88080664  9.17692782 99.26021541  1.00589327  0.30373803]


In [4]:
data_1e4.shape

(10000, 1000, 6)

In [3]:
data_1e5 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position/sbi-sim/data/sbi-benchmarks/odisseo_AllParametersPosition/x_100000.npy')
data_1e5 = data_1e5.reshape(-1, 6)
print(data_1e5.shape)

mean_x = np.mean(data_1e5, axis=0)
std_x = np.std(data_1e5, axis=0)

print("Mean and standard deviation for 1e4 point cloud data:")
print("Mean:", mean_x)
print("Standard Deviation:", std_x)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position/preprocess/mean_std_1e5_pointcloud.npz',
         mean_x=mean_x, std_x=std_x)

(100000000, 6)
Mean and standard deviation for 1e4 point cloud data:
Mean: [ 1.00883660e+01 -2.38331747e+01 -1.64147345e+00 -5.72392056e+01
 -7.34105222e-01 -2.77593317e-02]
Standard Deviation: [ 4.97251116 41.75389848  9.15409079 99.00183561  1.19778979  0.44866825]


# Point cloud new prior

In [2]:
import numpy as np

data_1e5 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position_newprior/sbi_sim/data/sbi-benchmarks/odisseo_AllParametersPosition_newprior/x_100000.npy')
data_1e5 = data_1e5.reshape(-1, 6)
print(data_1e5.shape)

mean_x = np.mean(data_1e5, axis=0)
std_x = np.std(data_1e5, axis=0)

print("Mean and standard deviation for 1et point cloud data:")
print("Mean:", mean_x)
print("Standard Deviation:", std_x)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position_newprior/preprocess/mean_std_1e5_pointcloud.npz',
         mean_x=mean_x, std_x=std_x)

(100000000, 6)
Mean and standard deviation for 1e4 point cloud data:
Mean: [ 9.67120027e+00 -2.51427313e+01 -9.83955404e-01 -5.92499524e+01
 -7.43672182e-01 -2.53998015e-02]
Standard Deviation: [ 3.80949835 35.21933473  7.2472688  97.30111818  0.36420859  0.1271499 ]


In [4]:
import numpy as np

data_1e5 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position_newprior/sbi_sim/data/sbi-benchmarks/odisseo_AllParametersPosition_newprior/theta_100000.npy')
print(data_1e5.shape)

mean_theta = np.mean(data_1e5, axis=0)
std_theta = np.std(data_1e5, axis=0)

print("Mean and standard deviation for 1e5 parameter data:")
print("Mean:", mean_theta)
print("Standard Deviation:", std_theta)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position_newprior/preprocess/mean_std_1e5_parameter.npz',
         mean_theta=mean_theta, std_theta=std_theta)

(100000, 13)
Mean and standard deviation for 1e5 parameter data:
Mean: [ 2.74980464e+00  3.75042429e+00  9.08632907e-03  1.16411493e+01
  1.72942957e+01  1.08334477e+01  3.14440629e+00  1.19960608e+01
  1.29600908e+00  6.99827628e+00  1.02473091e+02 -2.54985056e+02
 -9.99439000e+01]
Standard Deviation: [1.29996892e+00 4.32611823e-01 4.85451853e-03 1.30251602e-01
 7.23206504e+00 8.68653818e-02 9.72042465e-01 1.15726633e+00
 6.92391701e-01 5.76415741e-01 7.22851305e+00 1.44380159e+01
 1.15176129e+01]


# Point cloud uniform prior

In [2]:
import numpy as np

data_1e5 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position_uniform_prior/sbi_sim/data/sbi-benchmarks/odisseo_AllParametersPosition_uniformprior/x_200000.npy')
data_1e5 = data_1e5.reshape(-1, 6)
print(data_1e5.shape)

mean_x = np.mean(data_1e5, axis=0)
std_x = np.std(data_1e5, axis=0)

print("Mean and standard deviation for 2et point cloud data:")
print("Mean:", mean_x)
print("Standard Deviation:", std_x)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position_uniform_prior/preprocess/mean_std_2e5_pointcloud.npz',
         mean_x=mean_x, std_x=std_x)

(200000000, 6)
Mean and standard deviation for 2et point cloud data:
Mean: [ 9.97426338e+00 -2.39760185e+01 -1.17288800e+00 -5.73510025e+01
 -7.22468934e-01 -2.48103853e-02]
Standard Deviation: [4.23339599e+00 4.04460866e+01 8.29916567e+00 9.94215692e+01
 3.87777102e-01 8.15038269e-02]


In [1]:
import numpy as np

data_2e5 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position_uniform_prior/sbi_sim/data/sbi-benchmarks/odisseo_AllParametersPosition_uniformprior/theta_200000.npy')
print(data_2e5.shape)

mean_theta = np.mean(data_2e5, axis=0)
std_theta = np.std(data_2e5, axis=0)

print("Mean and standard deviation for 2e5 parameter data:")
print("Mean:", mean_theta)
print("Standard Deviation:", std_theta)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position_uniform_prior/preprocess/mean_std_2e5_parameter.npz',
         mean_theta=mean_theta, std_theta=std_theta)

(200000, 13)
Mean and standard deviation for 2e5 parameter data:
Mean: [ 2.74806226e+00  4.11412936e+00  1.00037063e-02  1.17082890e+01
  2.00028576e+01  1.09020971e+01  3.75213229e+00  1.20040885e+01
  1.30103410e+00  6.99711373e+00  1.02490139e+02 -2.54934084e+02
 -1.00001683e+02]
Standard Deviation: [1.29882413e+00 3.35340440e-01 3.46685602e-03 1.65798269e-01
 6.92655743e+00 1.65684731e-01 1.30181141e+00 1.15402678e+00
 6.93108675e-01 5.76412051e-01 7.21546124e+00 1.44406802e+01
 1.15490202e+01]


In [1]:
import numpy as np

data_1e5 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position_uniform_prior/sbi_sim/data/sbi-benchmarks/odisseo_AllParametersPosition_uniformprior/x_300000.npy')
data_1e5 = data_1e5.reshape(-1, 6)
print(data_1e5.shape)

mean_x = np.mean(data_1e5, axis=0)
std_x = np.std(data_1e5, axis=0)

print("Mean and standard deviation for 2et point cloud data:")
print("Mean:", mean_x)
print("Standard Deviation:", std_x)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position_uniform_prior/preprocess/mean_std_3e5_pointcloud.npz',
         mean_x=mean_x, std_x=std_x)


data_2e5 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position_uniform_prior/sbi_sim/data/sbi-benchmarks/odisseo_AllParametersPosition_uniformprior/theta_300000.npy')
print(data_2e5.shape)

mean_theta = np.mean(data_2e5, axis=0)
std_theta = np.std(data_2e5, axis=0)

print("Mean and standard deviation for 2e5 parameter data:")
print("Mean:", mean_theta)
print("Standard Deviation:", std_theta)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position_uniform_prior/preprocess/mean_std_3e5_parameter.npz',
         mean_theta=mean_theta, std_theta=std_theta)

(300000000, 6)
Mean and standard deviation for 2et point cloud data:
Mean: [ 9.97480502e+00 -2.39540084e+01 -1.17267395e+00 -5.74669762e+01
 -7.22734260e-01 -2.48342096e-02]
Standard Deviation: [4.23995634e+00 4.03776269e+01 8.29302692e+00 9.93784330e+01
 4.85463579e-01 9.50458794e-02]
(300000, 13)
Mean and standard deviation for 2e5 parameter data:
Mean: [ 2.74739053e+00  4.11572876e+00  1.00017012e-02  1.17080182e+01
  2.00030902e+01  1.09016327e+01  3.74861240e+00  1.20025631e+01
  1.30123668e+00  6.99748930e+00  1.02495133e+02 -2.54963348e+02
 -1.00010569e+02]
Standard Deviation: [1.29926380e+00 3.34568938e-01 3.46683750e-03 1.65779493e-01
 6.92679088e+00 1.66054410e-01 1.30073700e+00 1.15462511e+00
 6.92497844e-01 5.77523476e-01 7.21157495e+00 1.44378967e+01
 1.15534675e+01]


In [1]:
import numpy as np

data_1e5 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position_uniform_prior_TSIT5/sbi_sim/data/sbi-benchmarks/odisseo_AllParametersPosition_uniformprior_TSIT5/x_200000.npy')
data_1e5 = data_1e5.reshape(-1, 6)
print(data_1e5.shape)

mean_x = np.mean(data_1e5, axis=0)
std_x = np.std(data_1e5, axis=0)

print("Mean and standard deviation for 2et point cloud data:")
print("Mean:", mean_x)
print("Standard Deviation:", std_x)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position_uniform_prior_TSIT5/preprocess/mean_std_2e5_pointcloud.npz',
         mean_x=mean_x, std_x=std_x)


data_2e5 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position_uniform_prior_TSIT5/sbi_sim/data/sbi-benchmarks/odisseo_AllParametersPosition_uniformprior_TSIT5/theta_200000.npy')
print(data_2e5.shape)

mean_theta = np.mean(data_2e5, axis=0)
std_theta = np.std(data_2e5, axis=0)

print("Mean and standard deviation for 2e5 parameter data:")
print("Mean:", mean_theta)
print("Standard Deviation:", std_theta)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position_uniform_prior_TSIT5/preprocess/mean_std_2e5_parameter.npz',
         mean_theta=mean_theta, std_theta=std_theta)

(200000000, 6)
Mean and standard deviation for 2et point cloud data:
Mean: [ 1.00551178e+01 -2.49484962e+01 -1.18564474e+00 -5.44266593e+01
 -7.22744237e-01 -2.52998101e-02]
Standard Deviation: [  4.85158131  40.725307     8.37630125 100.97038057   0.56171632
   0.10438497]
(200000, 13)
Mean and standard deviation for 2e5 parameter data:
Mean: [ 2.75448502e+00  4.11377843e+00  9.99939988e-03  1.17083511e+01
  1.99938414e+01  1.09010526e+01  3.74805237e+00  1.20072137e+01
  1.30154036e+00  7.00014611e+00  1.02508790e+02 -2.55007318e+02
 -9.99787526e+01]
Standard Deviation: [1.30121987e+00 3.35542551e-01 3.46486878e-03 1.65641802e-01
 6.92934472e+00 1.65969809e-01 1.29933535e+00 1.15512574e+00
 6.92400542e-01 5.76424381e-01 7.22075220e+00 1.44351500e+01
 1.15527563e+01]


# Fix position TSIT5

In [1]:
import numpy as np

data_1e5 = np.load('/export/data/vgiusepp/odisseo_data/data_fix_position_uniform_prior_TSTIT5/sbi_sim/data/sbi-benchmarks/odisseo_AllParameters_fixposition_uniformprior_TSIT5/x_100000.npy')
data_1e5 = data_1e5.reshape(-1, 6)
print(data_1e5.shape)

mean_x = np.mean(data_1e5, axis=0)
std_x = np.std(data_1e5, axis=0)

print("Mean and standard deviation for 1et point cloud data:")
print("Mean:", mean_x)
print("Standard Deviation:", std_x)
np.savez('/export/data/vgiusepp/odisseo_data/data_fix_position_uniform_prior_TSTIT5/preprocess/mean_std_1e5_pointcloud.npz',
         mean_x=mean_x, std_x=std_x)


data_2e5 = np.load('/export/data/vgiusepp/odisseo_data/data_fix_position_uniform_prior_TSTIT5/sbi_sim/data/sbi-benchmarks/odisseo_AllParameters_fixposition_uniformprior_TSIT5/theta_100000.npy')
print(data_2e5.shape)

mean_theta = np.mean(data_2e5, axis=0)
std_theta = np.std(data_2e5, axis=0)

print("Mean and standard deviation for 1e5 parameter data:")
print("Mean:", mean_theta)
print("Standard Deviation:", std_theta)
np.savez('/export/data/vgiusepp/odisseo_data/data_fix_position_uniform_prior_TSTIT5/preprocess/mean_std_1e5_parameter.npz',
         mean_theta=mean_theta, std_theta=std_theta)

(100000000, 6)
Mean and standard deviation for 1et point cloud data:
Mean: [ 8.81313618e+00 -2.80738900e+01 -1.29987301e+00 -3.75665260e+01
 -8.59364409e-01 -2.33847081e-02]
Standard Deviation: [ 3.7174879  34.22204908  3.45570379 98.33168496  0.39040496  0.13065352]
(100000, 7)
Mean and standard deviation for 1e5 parameter data:
Mean: [11.70792268  1.27186978 10.90119012  0.54435629 -0.48624553  9.72036555
  0.34564094]
Standard Deviation: [0.16563681 0.16552091 0.16605304 0.16569611 0.16613399 0.16595909
 0.16630976]


In [5]:
import numpy as np

data_1e5 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position_fixed_time_uniform_prior_TSTIT5/sbi_sim/data/sbi-benchmarks/data_varying_position_fixed_time_uniform_prior_TSTIT5/x_100000.npy')
data_1e5 = data_1e5.reshape(-1, 6)
print(data_1e5.shape)

mean_x = np.mean(data_1e5, axis=0)
std_x = np.std(data_1e5, axis=0)

print("Mean and standard deviation for 1et point cloud data:")
print("Mean:", mean_x)
print("Standard Deviation:", std_x)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position_fixed_time_uniform_prior_TSTIT5/preprocess/mean_std_1e5_pointcloud.npz',
         mean_x=mean_x, std_x=std_x)


data_2e5 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position_fixed_time_uniform_prior_TSTIT5/sbi_sim/data/sbi-benchmarks/data_varying_position_fixed_time_uniform_prior_TSTIT5/theta_100000.npy')
print(data_2e5.shape)

mean_theta = np.mean(data_2e5, axis=0)
std_theta = np.std(data_2e5, axis=0)

print("Mean and standard deviation for 1e5 parameter data:")
print("Mean:", mean_theta)
print("Standard Deviation:", std_theta)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position_fixed_time_uniform_prior_TSTIT5/preprocess/mean_std_1e5_parameter.npz',
         mean_theta=mean_theta, std_theta=std_theta)

(100000000, 6)
Mean and standard deviation for 1et point cloud data:
Mean: [ 9.47786669e+00 -2.53725943e+01 -8.93549034e-01 -5.90537292e+01
 -7.65461723e-01 -2.73231068e-02]
Standard Deviation: [ 3.90811442 32.98187433  7.20484335 94.18409939  0.5420575   0.11962076]
(100000, 13)
Mean and standard deviation for 1e5 parameter data:
Mean: [  11.70749189    1.27037038   10.90185924    0.54333498   -0.48473719
    9.72043133    0.34575433   11.99502394    1.30061923    6.99856429
  102.46353643 -254.96949307  -99.96090678]
Standard Deviation: [ 0.1660873   0.16591431  0.16558435  0.16620194  0.16565113  0.16583343
  0.165916    1.15828043  0.69313323  0.57740235  7.23140733 14.41795552
 11.56425551]


In [1]:
import numpy as np
data_2e5 = np.load('/export/data/vgiusepp/odisseo_data/data_varying_position_fixed_time_uniform_prior_TSTIT5/sbi_sim/data/sbi-benchmarks/data_varying_position_fixed_time_uniform_prior_TSTIT5/theta_100000.npy')
print(data_2e5.shape)
data_2e5[:, :7] = 10**data_2e5[:, :7]

mean_theta = np.mean(data_2e5, axis=0)
std_theta = np.std(data_2e5, axis=0)

print("Mean and standard deviation for 1e5 parameter data:")
print("Mean:", mean_theta)
print("Standard Deviation:", std_theta)
np.savez('/export/data/vgiusepp/odisseo_data/data_varying_position_fixed_time_uniform_prior_TSTIT5/preprocess/mean_std_1e5_parameter_nolog.npz',
         mean_theta=mean_theta, std_theta=std_theta)

(100000, 13)
Mean and standard deviation for 1e5 parameter data:
Mean: [ 5.45836271e+11  1.99485616e+01  8.53580013e+10  3.74087428e+00
  3.50476716e-01  5.62240562e+09  2.37288268e+00  1.19950239e+01
  1.30061923e+00  6.99856429e+00  1.02463536e+02 -2.54969493e+02
 -9.99609068e+01]
Standard Deviation: [1.89372250e+11 6.92491252e+00 2.95150741e+10 1.30048759e+00
 1.21149729e-01 1.94860378e+09 8.22882689e-01 1.15828043e+00
 6.93133228e-01 5.77402354e-01 7.23140733e+00 1.44179555e+01
 1.15642555e+01]


# Galax

In [1]:
import numpy as np


data_1e6 = np.load('/export/data/vgiusepp/galax_data/data_varying_position_uniform_prior/sbi_sim/data/sbi-benchmarks/galax_AllParametersPosition_uniformprior/theta_1000000.npy')
print(data_1e6.shape)

mean_theta = np.mean(data_1e6, axis=0)
std_theta = np.std(data_1e6, axis=0)

print("Mean and standard deviation for 1e6 parameter data:")
print("Mean:", mean_theta)
print("Standard Deviation:", std_theta)
np.savez('/export/data/vgiusepp/galax_data/data_varying_position_uniform_prior/preprocess/mean_std_1e6_parameter.npz',
         mean_theta=mean_theta, std_theta=std_theta)

(1000000, 12)
Mean and standard deviation for 1e6 parameter data:
Mean: [ 1.6320800e+04  5.4590849e+11  2.0001202e+01  8.5247336e+10
  3.7504582e+00  3.5013422e-01  1.2000294e+01  1.3002234e+00
  7.0002398e+00  1.0248079e+02 -2.5489001e+02 -9.9986877e+01]
Standard Deviation: [8.8398154e+03 1.8918631e+11 6.9269714e+00 2.9494409e+10 1.2995077e+00
 1.2123106e-01 1.1535504e+00 6.9313234e-01 5.7742172e-01 7.2149596e+00
 1.4425357e+01 1.1549500e+01]


In [4]:
data_1e6 = np.load('/export/data/vgiusepp/galax_data/data_varying_position_uniform_prior/sbi_sim/data/sbi-benchmarks/galax_AllParametersPosition_uniformprior/x_1000000.npy')
print(data_1e6.shape)

data_1e6 = data_1e6.reshape(-1, 6)

mean_x = np.mean(data_1e6, axis=0)
std_x = np.std(data_1e6, axis=0)

print("Mean and standard deviation for 1e6 parameter data:")
print("Mean:", mean_x)
print("Standard Deviation:", std_x)
np.savez('/export/data/vgiusepp/galax_data/data_varying_position_uniform_prior/preprocess/mean_std_1e6_pointcloud.npz',
         mean_x=mean_x, std_x=std_x)

(1000000, 200, 6)
Mean and standard deviation for 1e6 parameter data:
Mean: [  1.5269282    0.63246214   0.16777216 -21.47526     -6.1715417
  -5.3689384 ]
Standard Deviation: [ 4.0537677  1.1703541  0.3199752 63.8563    18.650385  10.951647 ]


In [3]:
mean_x.shape

(200, 6)